# 02 — Counts + Periods (Stages 2–3)
**Does:** monthly inventory WITHOUT saving text, Tier triage, and — behind an explicit freeze gate — the frozen `period_definitions.csv` v1.0.
**Design (audit-driven):** bulk dumps are torrent-only upstream (not Colab-viable), the HF mirror id is unverified, and the aggregate endpoint times out on large windows. So: **aggregate-first per sub** (1 cheap query per sub×type, auto-splitting full→yearly→quarterly on timeout), token estimates via a measured per-sub tokens/doc factor from a bounded sample, method recorded per row. Full-range census paging is NOT attempted (infeasible on a best-effort free API) — estimates are labeled, never silently promoted.
**Gate:** cells run in order; the freezer refuses to run until `FREEZE_PERIODS=True` AND you have reviewed the triage table.


In [ ]:
# Cell 1 — COUNT PLAN & MODE SELECTION (the only cell you must edit).
# Choose between training on the ENTIRETY OF REDDIT or a SPECIFIC SET OF SUBREDDITS.
CORPUS_MODE = "SUBREDDIT_LIST"   # Options: "ALL_REDDIT" (entirety of reddit) or "SUBREDDIT_LIST" (selected subreddits)

# If CORPUS_MODE == "SUBREDDIT_LIST": define your subreddits here, or set to None to load from config/subreddit_list.csv
CUSTOM_SUBREDDITS = None   # None = load from config/subreddit_list.csv; or provide list: ["AskAcademia", "PhD"]

FULL_RANGE = False        # False = probe sample months (proves path); True = full 2013-01..2025-12
PROBE_MONTHS = ["2015-06", "2019-01", "2023-07"]
FREEZE_PERIODS = False    # keep False until triage reviewed; True builds period_definitions.csv
ALLOW_ESTIMATES = True    # freezer allows estimate-based counts

print(f"Mode: {CORPUS_MODE} | Range: {'FULL (2013-2025)' if FULL_RANGE else PROBE_MONTHS} | Freeze: {FREEZE_PERIODS}")


In [ ]:
# Cell 2 — Setup: root, config v0.3.x, logger, manifests, dirs.
import os, sys, csv, json, re, time, hashlib, datetime, statistics
from pathlib import Path
from collections import defaultdict
import requests, yaml
from src.paths import get_project_root
from src.storage import atomic_write_text, sha256_file
from src.manifests import RETRIEVAL_COLS, load_manifest, upsert_manifest_row
from src.api import api_get, parse_agg

ROOT = get_project_root()
cfg = yaml.safe_load(open(ROOT / "config/project_config.yaml", encoding="utf-8"))
ver = tuple(int(x) for x in cfg["config_version"].lstrip("v").split("."))
assert ver >= (0, 3, 0), f"need config v0.3.0+, found {cfg['config_version']}"
CFG_SHA = sha256_file(ROOT / "config/project_config.yaml")

P, SE = cfg["periodization"], cfg.get("seed_eval", {})
STD, FLOOR, AXIS = P["min_usable_tokens_per_model"], P["absolute_floor_tokens"], P["axis_grade_tokens"]
BUDGET = cfg["reference_sampling"]["within_period_cap"]["max_train_tokens_per_model"]
print(f"config {cfg['config_version']} thresholds std={STD} floor={FLOOR} axis={AXIS} cap={BUDGET}")

import logging
ts = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
LOGP = ROOT / f"logs/02_counts_periods__{ts}__cfg-{cfg['config_version']}.log"
LOGP.parent.mkdir(parents=True, exist_ok=True)
lg = logging.getLogger("c2p"); lg.setLevel(logging.INFO); lg.handlers.clear()
fh = logging.FileHandler(LOGP); fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
sh = logging.StreamHandler(sys.stdout); sh.setLevel(logging.WARNING)
lg.addHandler(fh); lg.addHandler(sh)

for d in ["metadata/monthly_counts", "metadata/coverage_reports", "metadata/corpus_statistics", "diagnostics/counts"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

CMAN = ROOT / "manifests/retrieval_manifest.csv"
if not CMAN.exists():
    atomic_write_text(CMAN, ",".join(RETRIEVAL_COLS) + "\n")

def mrows(): return load_manifest(CMAN)
def mupsert(row): upsert_manifest_row(CMAN, row, ["unit_id"], RETRIEVAL_COLS)
def atomic_text(path: Path, text: str): atomic_write_text(path, text)
today = datetime.datetime.now(datetime.timezone.utc).date().isoformat()
print("setup ready (imported src modules)")
# Resolve Target Subreddits based on CORPUS_MODE
if CORPUS_MODE == "ALL_REDDIT":
    CANDIDATE_SUBS = ["ALL_REDDIT"]
else:
    if CUSTOM_SUBREDDITS is not None and len(CUSTOM_SUBREDDITS) > 0:
        CANDIDATE_SUBS = list(CUSTOM_SUBREDDITS)
    else:
        sub_list_file = ROOT / "config/subreddit_list.csv"
        if sub_list_file.exists():
            rows = [r for r in csv.DictReader(open(sub_list_file, encoding="utf-8")) if r.get("include", "").strip().upper() == "TRUE"]
            CANDIDATE_SUBS = [r["subreddit"] for r in rows]
        else:
            CANDIDATE_SUBS = ["AskAcademia", "PhD", "academia"]
print(f"Target subreddits/scope ({len(CANDIDATE_SUBS)}): {CANDIDATE_SUBS}")


In [ ]:
# Cell 3 — AGGREGATE COUNTING: one cheap query per (sub, type); auto-split full→yearly→quarterly on timeout.
# Intermediate counts are cached in metadata/monthly_counts/ so resume NEVER loses data.
def ts_to_month(v):
    try:
        n = int(v)
        return datetime.datetime.fromtimestamp(n, datetime.timezone.utc).strftime("%Y-%m")
    except (ValueError, TypeError):
        return str(v)[:7]

def agg_window(sub, ctype, after, before, freq):
    ep = "/posts/search/aggregate" if ctype == "submissions" else "/comments/search/aggregate"
    params = {"aggregate": "created_utc", "frequency": freq, "after": after, "before": before}
    if sub != "ALL_REDDIT": params["subreddit"] = sub
    r = api_get(ep, params, tries=2, timeout=90, logger=lg)
    return parse_agg(r.json())

def count_sub(sub, ctype):
    'Returns (month->docs dict, method string). Tries full range, then yearly, then quarterly windows.'
    y0, y1 = 2013, 2025
    try:
        rows = agg_window(sub, ctype, f"{y0}-01-01", f"{y1+1}-01-01", "month")
        return {ts_to_month(t): c for t, c in rows}, "aggregate:month:full-range"
    except Exception as e1:
        lg.warning(f"{sub}/{ctype} full-range failed ({str(e1)[:80]}); splitting yearly")
    out, wins = {}, []
    for y in range(y0, y1 + 1):
        try:
            rows = agg_window(sub, ctype, f"{y}-01-01", f"{y+1}-01-01", "month")
            for t, c in rows: out[ts_to_month(t)] = out.get(ts_to_month(t), 0) + c
            wins.append(str(y))
        except Exception:
            for q in range(1, 5):
                m0 = (q - 1) * 3 + 1
                m1 = m0 + 3
                a, b = f"{y}-{m0:02d}-01", f"{y+1 if m1 > 12 else y}-{m1 if m1 <= 12 else 1:02d}-01"
                try:
                    rows = agg_window(sub, ctype, a, b, "month")
                    for t, c in rows: out[ts_to_month(t)] = out.get(ts_to_month(t), 0) + c
                    wins.append(f"{y}Q{q}")
                except Exception as e3:
                    lg.error(f"{sub}/{ctype} {y}Q{q} failed: {str(e3)[:100]}")
    return out, f"aggregate:split[{','.join(wins)}]"

def sample_factor(sub, ctype, pages=3):
    'Bounded sample (<=pages*100 docs, fields-trimmed) -> median tokens/doc + valid rate.'
    ep = "/posts/search" if ctype == "submissions" else "/comments/search"
    toks, valid, n = [], 0, 0
    after = int(datetime.datetime(2019, 1, 1, tzinfo=datetime.timezone.utc).timestamp())
    try:
        for _ in range(pages):
            params = {"after": after, "limit": 100, "sort": "asc", "fields": "id,created_utc,subreddit,body,title,selftext"}
            if sub != "ALL_REDDIT": params["subreddit"] = sub
            r = api_get(ep, params, tries=3, timeout=45, logger=lg)
            batch = r.json().get("data", [])
            if not batch: break
            for rec in batch:
                n += 1
                txt = rec.get("body", "") if ctype == "comments" else ((rec.get("title", "") or "") + "\n" + (rec.get("selftext", "") or ""))
                s = (txt or "").strip()
                if s in ("[deleted]", "[removed]", "") or len(s.split()) < 3: continue
                valid += 1; toks.append(len(s.split()))
            after = int(batch[-1].get("created_utc", after)) + 1
            if len(batch) < 100: break
    except Exception as e:
        lg.error(f"factor sample {sub}/{ctype}: {str(e)[:120]}")
    if not toks: return 25.0, 0.8, 0, "fallback_defaults_unmeasured"
    return float(statistics.median(toks)), valid / max(1, n), n, f"measured_n={n}"

counts_dir = ROOT / "metadata/monthly_counts"
counts, methods, factors = {}, {}, {}
done = skip = fail = 0

for sub in CANDIDATE_SUBS:
    for ctype in ["comments", "submissions"]:
        unit = f"countagg__{sub}__{ctype}"
        cache_file = counts_dir / f"{sub.lower()}__{ctype}.json"
        prior = {r["unit_id"]: r for r in mrows()}.get(unit)
        
        # Safe resume: reload counts & factors from intermediate cache file
        if prior and prior["status"] == "complete" and cache_file.exists():
            try:
                cached = json.loads(cache_file.read_text(encoding="utf-8"))
                counts[(sub, ctype)] = cached["counts"]
                methods[(sub, ctype)] = cached.get("method", prior.get("last_cursor", "cached"))
                factors[(sub, ctype)] = (float(cached["median"]), float(cached["valid_rate"]), cached.get("factor_source", "cached"))
                skip += 1
                print(f"SKIP (reloaded) {sub}/{ctype}: {len(cached['counts'])} months docs={sum(cached['counts'].values())}")
                continue
            except Exception as ce:
                lg.warning(f"Cache reload failed for {cache_file}, will re-query: {ce}")

        row = {"unit_id": unit, "source": "arctic_shift_api", "source_url_or_query": "aggregate=created_utc,frequency=month",
               "subreddit": sub, "start_ts": "2013-01-01T00:00:00Z", "end_ts": "2026-01-01T00:00:00Z",
               "content_type": ctype, "status": "in_progress",
               "attempt_count": int((prior or {}).get("attempt_count", 0)) + 1,
               "started_at": datetime.datetime.now(datetime.timezone.utc).isoformat(), "completed_at": "",
               "last_cursor": "", "n_read": 0, "n_usable": 0, "token_estimate": 0, "output_tmp": "",
               "output_final": str(cache_file), "output_sha256": "", "error_category": "", "error_excerpt": "",
               "config_version": cfg["config_version"], "config_sha256": CFG_SHA, "retrieval_date": today}
        mupsert(row)
        try:
            mc, meth = count_sub(sub, ctype)
            med, vrate, sn, fsrc = sample_factor(sub, ctype)
            counts[(sub, ctype)] = mc; methods[(sub, ctype)] = meth
            factors[(sub, ctype)] = (med, vrate, fsrc)
            tot = sum(mc.values())
            # Persist intermediate cache to disk
            cache_data = {"subreddit": sub, "content_type": ctype, "counts": mc,
                          "median": med, "valid_rate": vrate, "factor_source": fsrc,
                          "method": meth, "updated_at": today}
            atomic_text(cache_file, json.dumps(cache_data, indent=2))
            c_sha = sha256_file(cache_file)
            row.update({"status": "complete", "completed_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
                        "n_read": tot, "n_usable": int(tot * vrate), "token_estimate": int(tot * vrate * med),
                        "output_final": str(cache_file), "output_sha256": c_sha,
                        "last_cursor": meth}); mupsert(row); done += 1
            print(f"OK {sub}/{ctype}: {len(mc)} months docs={tot} tok/doc~{med:.0f} [{meth[:40]}]")
        except Exception as e:
            lg.exception(unit)
            row.update({"status": "failed", "error_category": "api", "error_excerpt": str(e)[:300]}); mupsert(row); fail += 1
            print(f"FAIL {sub}/{ctype}: {str(e)[:120]}")

print(f"count units done={done} skipped={skip} failed={fail}")
atomic_text(ROOT / "diagnostics/counts/count_factors.csv",
            "sub,ctype,median_tok_per_doc,valid_rate,source\n" +


In [ ]:
# Cell 4 — MONTHLY ROLLUP (the big count CSV): docs + token estimates + method per row.
def months_wanted():
    if FULL_RANGE:
        ms, cur = [], "2013-01"
        while cur <= "2025-12":
            ms.append(cur)
            y, m = int(cur[:4]), int(cur[5:]) + 1
            if m == 13: y, m = y + 1, 1
            cur = f"{y:04d}-{m:02d}"
        return ms
    return list(PROBE_MONTHS)
MONTHS = months_wanted()
STAT_COLS = ["month","subreddit","content_type","n_docs","valid_rate","tok_per_doc_median","token_est",
             "method","factor_source","status","config_version"]
lines, warns = [",".join(STAT_COLS)], []
for sub in CANDIDATE_SUBS:
    for ctype in ["comments", "submissions"]:
        mc = counts.get((sub, ctype), {})
        med, vrate, fsrc = factors.get((sub, ctype), (25.0, 0.8, "fallback"))
        meth = methods.get((sub, ctype), "skipped_or_failed")
        for mm in MONTHS:
            d = mc.get(mm, 0)
            lines.append(",".join(str(x) for x in [mm, sub, ctype, d, f"{vrate:.2f}", f"{med:.1f}",
                int(d * vrate * med), meth, fsrc, "complete" if (sub, ctype) in counts else "missing", cfg["config_version"]]))
        if sum(mc.get(mm, 0) for mm in MONTHS) == 0 and (sub, ctype) in counts:
            warns.append(f"{sub}/{ctype}: zero docs in counted months — check sub name / coverage")
ROLL = ROOT / "metadata/monthly_rollup.csv"
atomic_text(ROLL, "\n".join(lines) + "\n")
print(f"monthly_rollup.csv: {len(lines)-1} rows -> {ROLL}")
for w in warns: print("WARN", w)


In [ ]:
# Cell 5 — TIER TRIAGE: apply the well-represented rule mechanically (proposal; freezer uses it).
def quarter_of(mm): return mm[:4] + "Q" + str((int(mm[5:]) - 1) // 3 + 1)
qtok = defaultdict(int)
for sub in CANDIDATE_SUBS:
    for ctype in ["comments", "submissions"]:
        mc = counts.get((sub, ctype), {})
        med, vrate, _ = factors.get((sub, ctype), (25.0, 0.8, "fallback"))
        for mm in MONTHS:
            qtok[(sub, quarter_of(mm))] += int(mc.get(mm, 0) * vrate * med)
W = cfg["scope"]["well_represented_rule_provisional"]
tri = []
for sub in CANDIDATE_SUBS:
    active = [mm for mm in MONTHS if sum(counts.get((sub, c), {}).get(mm, 0) for c in ["comments", "submissions"]) >= 100]
    a_share = len(active) / max(1, len(MONTHS))
    qs = sorted({quarter_of(mm) for mm in MONTHS})
    suff = [q for q in qs if sum(qtok.get((sub, q), 0) for _ in [0]) >= STD]
    s_share = len(suff) / max(1, len(qs))
    gaps, run = 0, 0
    for mm in MONTHS:
        if mm in active: run = 0
        else: run += 1; gaps = max(gaps, run)
    tier = "solo_per_period" if (a_share >= W["active_month_share_min"] and s_share >= W["sufficient_period_share_min"] and gaps <= W["max_gap_months"]) else ("pooled_only" if a_share >= 0.5 else "excluded")
    tri.append((sub, tier, round(a_share, 2), round(s_share, 2), gaps))
    print(f"{sub:15s} {tier:16s} active={a_share:.2f} sufficient={s_share:.2f} maxgap={gaps}")
atomic_text(ROOT / "metadata/tier_assignments.csv",
            "subreddit,tier,active_share,sufficient_share,max_gap_months,config_version\n" +
            "".join(f"{s},{t},{a},{ss},{g},{cfg['config_version']}\n" for s, t, a, ss, g in tri))
print("REVIEW this table. Freezer (Cell 6) stays locked until FREEZE_PERIODS=True.")


In [ ]:
# Cell 6 — FREEZE GATE: build period_definitions.csv v1.0 from MONTHLY atoms (never pre-quartered).
# Merge-forward in month units (base 3 tracked / 6 reference, cap 12); overflow -> fraction. Real dates out.
if not FREEZE_PERIODS:
    print("LOCKED: set FREEZE_PERIODS=True in Cell 1 after reviewing triage, then rerun from here.")
else:
    def month_add(mm, k):
        y, m = int(mm[:4]), int(mm[5:]) + k
        while m > 12: y, m = y + 1, m - 12
        while m < 1: y, m = y - 1, m + 12
        return f"{y:04d}-{m:02d}"
    def span_id(s, e):
        q = lambda mm: mm[:4] + "q" + str((int(mm[5:]) - 1) // 3 + 1)
        return q(s) if month_add(s, 3) == e and s[5:] in ("01", "04", "07", "10") else s + "_" + month_add(e, -1)
    def freeze_months(items, base):
        """items: sorted [(month, est_tokens)]. Returns period dicts with real dates."""
        out, i = [], 0
        while i < len(items):
            s = items[i][0]; t, n = 0, 0
            while n < base and i + n < len(items): t += items[i + n][1]; n += 1
            while t < STD and n < 12 and i + n < len(items): t += items[i + n][1]; n += 1
            e = month_add(s, n)
            suff = "axis_grade" if t >= AXIS else ("sufficient" if t >= STD else ("marginal_merge_first" if t >= FLOOR else "below_floor_pool_only"))
            reason = "base_cell_sufficient" if (t >= STD and n == base) else (f"merged_{n}mo_below_threshold" if t >= STD else "insufficient_marked")
            frac = min(1.0, BUDGET / t) if t > 0 else 1.0
            out.append((s, e, t, int(t * frac), round(frac, 4), reason, suff, n))
            i += n
        return out
    sub_months = {}
    for sub in CANDIDATE_SUBS:
        d = defaultdict(int)
        for ctype in ["comments", "submissions"]:
            mc = counts.get((sub, ctype), {})
            med, vrate, _ = factors.get((sub, ctype), (25.0, 0.8, "fallback"))
            for mm in MONTHS:
                d[mm] += int(mc.get(mm, 0) * vrate * med)
        sub_months[sub] = sorted(d.items())
    ref_months = sorted(((mm, sum(dict(sub_months[s]).get(mm, 0) for s in CANDIDATE_SUBS)) for mm in MONTHS))
    PCOLS = ["model_id","corpus_type","subreddit_or_group","start_date","end_date","est_docs","est_tokens",
             "total_eligible_tokens","sampled_train_tokens","sampling_fraction","boundary_reason","sufficiency","method","config_version"]
    prows = []
    for sub in CANDIDATE_SUBS:
        if not sub_months[sub]: continue
        for (s, e, t, st, fr, rs, sf, n) in freeze_months(sub_months[sub], 3):
            mid = f"w2v__{sub.lower()}__{span_id(s, e)}"
            prows.append([mid, "tracked", sub, s + "-01", e + "-01", "", t, t, st, fr, rs, sf, "aggregate+factors", cfg["config_version"]])
    if ref_months:
        for (s, e, t, st, fr, rs, sf, n) in freeze_months(ref_months, 6):
            mid = f"w2v__refpooled__{span_id(s, e)}"
            prows.append([mid, "reference", "REF_POOLED", s + "-01", e + "-01", "", t, t, st, fr, rs, sf, "aggregate+factors", cfg["config_version"]])
    PDEF = ROOT / "config/period_definitions.csv"
    tmp = PDEF.with_suffix(".tmp")
    f = open(tmp, "w", newline="", encoding="utf-8"); w = csv.writer(f); w.writerow(PCOLS); w.writerows(prows)
    f.flush(); os.fsync(f.fileno()); f.close(); os.replace(tmp, PDEF)
    print(f"FROZEN: {len(prows)} period rows -> {PDEF} (config {cfg['config_version']})")
    for r in prows: print(f"  {r[11]:22s} {r[0]} {r[3]}..{r[4]} tok~{r[7]} frac={r[9]} ({r[10]})")


In [ ]:
# Cell 7 — VALIDATION + END-OF-RUN SUMMARY.
print("=" * 70)
print(f"COUNTS done | rollup rows={len(lines)-1} | triage subs={len(tri)} | freeze={'ON' if FREEZE_PERIODS else 'OFF (review triage first)'}")
print("wrote: metadata/monthly_rollup.csv, metadata/tier_assignments.csv, diagnostics/counts/count_factors.csv")
print("rerun safe: YES (complete count units skip; freezer rewrites atomically)")
print("next: " + ("03_build_shards.ipynb" if FREEZE_PERIODS else "review triage, then rerun with FREEZE_PERIODS=True"))
print("=" * 70)
